# A/B Test Analysis Notebook

Этот ноутбук демонстрирует полный рабочий процесс анализа A/B-теста, включая загрузку данных, предварительную обработку, агрегирование, визуализацию и экспорт результатов.

In [ ]:
# Импорт необходимых библиотек
import sqlite3
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")


In [ ]:
# Загрузка и обзор данных
db_path = Path("data/ab_test.db")
with sqlite3.connect(db_path) as conn:
    df = pd.read_sql_query("SELECT * FROM raw_ab_test_events", conn)

print("Rows:", len(df))
print("Columns:", list(df.columns))
df.head()

In [ ]:
# Предварительная обработка данных
df["event_date"] = pd.to_datetime(df["event_date"], errors="coerce")
df["variation"] = df["variation"].astype(str)
df["event_type"] = df["event_type"].astype(str)
df["page"] = df["page"].astype(str)
df["country"] = df["country"].astype(str)

df_info = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum().values,
    "unique": [df[col].nunique(dropna=True) for col in df.columns],
})
df_info

In [ ]:
# Анализ и агрегирование
df["is_purchase"] = df["event_type"] == "purchase"

variation_summary = (
    df.groupby("variation")
    .agg(
        users=("user_id", pd.Series.nunique),
        conversions=("is_purchase", "sum"),
        total_revenue=("revenue", "sum"),
    )
    .reset_index()
)
variation_summary["conversion_rate"] = (variation_summary["conversions"] / variation_summary["users"]).round(4)
variation_summary["revenue_per_user"] = (variation_summary["total_revenue"] / variation_summary["users"]).round(2)
variation_summary

In [ ]:
# Визуализация результатов
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.barplot(data=variation_summary, x="variation", y="conversion_rate", ax=axes[0])
axes[0].set_title("Conversion Rate by Variation")

page_summary = (
    df.groupby(["variation", "page"])    
    .agg(users=("user_id", pd.Series.nunique), purchases=("is_purchase", "sum"))
    .reset_index()
)
page_summary["conversion_rate"] = page_summary["purchases"] / page_summary["users"]

sns.barplot(data=page_summary, x="page", y="conversion_rate", hue="variation", ax=axes[1])
axes[1].set_title("Conversion Rate by Page and Variation")
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45)

country_summary = (
    df.groupby(["variation", "country"])    
    .agg(users=("user_id", pd.Series.nunique), purchases=("is_purchase", "sum"))
    .reset_index()
)
country_summary["conversion_rate"] = country_summary["purchases"] / country_summary["users"]

sns.barplot(data=country_summary, x="country", y="conversion_rate", hue="variation", ax=axes[2])
axes[2].set_title("Conversion Rate by Country and Variation")
axes[2].set_xticklabels(axes[2].get_xticklabels(), rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Сохранение и экспорт
output_dir = Path("reports")
output_dir.mkdir(parents=True, exist_ok=True)
variation_summary.to_csv(output_dir / "notebook_variation_summary.csv", index=False)
page_summary.to_csv(output_dir / "notebook_page_summary.csv", index=False)
country_summary.to_csv(output_dir / "notebook_country_summary.csv", index=False)
print(f"Saved notebook summary outputs to {output_dir}")